In [19]:
import pandas as pd
import pyecap
import os
from pathlib import Path
import scipy.io as sio
import numpy as np
import pyecap.io.adinstruments_io
import ast
import matplotlib.pyplot as plt
import sys

from pyecap.io.adinstruments_io import AdInstrumentsIO, convert_time, ADInstrumentsBin
import datetime
import plotly.graph_objects as go
import plotly.io as pio
#from plotly_resampler import FigureResampler, FigureWidgetResampler
from scipy import signal, fft
from scipy.optimize import curve_fit
import seaborn as sns

In [20]:
""" Sigmoid Function Definitions """
def sigmoid(x, y_min, y_max, slope, inf_pt):
    return y_max / (1 + np.exp(-slope * (x - inf_pt))) + y_min

def sigmoid_log10(x, y_min, y_max, slope, inf_pt):
    return y_max / (1 + 10**(-slope * (x - inf_pt))) + y_min

def sigmoid_fit(dataDF, y_variable, init_params = [1,1,100,0], bounds = ([0, 0, 0, -np.inf], [np.inf, np.inf, np.inf, np.inf]), log_transform = False, pad_min=False):
    y_max, slope, inf_pt, y_min = init_params

    x_d = np.array(dataDF['pulse amplitude (μA)'])
    y_d = np.array(dataDF[y_variable])
    y_max = np.max(y_d)
    #y_max = np.median(dataDF.loc[ dataDF['pulse amplitude (μA)'] >= 2000, y_variable])

    if pad_min:
        x_d = np.insert(x_d, 0,10)
        y_d = np.insert(y_d, 0,0)

    if log_transform:
        x_d = np.log10(x_d)
        inf_pt = np.log10(inf_pt)
        popt,pcov = curve_fit(sigmoid_log10, xdata = x_d, ydata = y_d, p0 = [y_max, slope, inf_pt, y_min], bounds = bounds, max_nfev=10000)
    else:
        popt,pcov = curve_fit(sigmoid, xdata = x_d, ydata = y_d, p0 = [y_max, slope, inf_pt, y_min], bounds = bounds, max_nfev=10000)

    # print(x_d)
    # print(y_max)

    return popt,pcov

In [ ]:

filePATH = r'G:\Data\EN_Mapping\LM6_190529\pnpig_190529-190529-124227'

"""Import Stimulation data"""
stim_data = pyecap.Stim(filePATH)
stimDF = stim_data.parameters

"""Instantiate pyecap objects"""
emg_drop_ch = ['Wav1 ' + str(ch) for ch in [1,2,3,4,11,12,13,14,15,16]]

raw_ephys = pyecap.Ephys(filePATH, stores = 'Wav1')
raw_emg = raw_ephys.remove_ch(emg_drop_ch)
raw_emg = raw_emg.channel_reference(channel = 'Wav1 7')

# Filtering
raw_emg_f = raw_emg.filter_gaussian(Wn = 3000, btype='lowpass')
raw_emg_f = raw_emg_f.filter_powerline()
raw_emg_f = raw_emg_f.filter_median(btype='highpass')

EMGf = pyecap.EMG(raw_emg_f, stim_data)

rms = EMGf.pulse_RMS(parameters = stimDF.index.tolist(), channels = 'Wav1 8', window = (EMGf._time_to_index))

In [ ]:
""" In rms, dim 1 = pulse #, dim 2 = channel 
X data = pulse amplitude
Y data = RMS based on calculation

"""

x = rms[(0,0)]

In [27]:
EMGf._time_to_index(6, units = 'milliseconds')

np.int64(146)

In [6]:
x[0]

array([0.00010893])

In [7]:
x[0][0]

np.float64(0.00010893032031456791)

In [9]:
m[(0,0)]

array([[ 7.62753322e-06,  9.45133867e-06,  1.10913525e-05,
         1.27425708e-05,  1.42385602e-05,  1.58600972e-05,
         1.76078323e-05,  1.94176188e-05,  2.13249705e-05,
         2.31005435e-05,  2.48514663e-05,  2.65110105e-05,
         2.81191977e-05,  2.96619009e-05,  3.11663236e-05,
         3.27016872e-05,  3.43241686e-05,  3.60118191e-05,
         3.78998607e-05,  3.97424058e-05,  4.19782598e-05,
         4.73094281e-05,  6.79722008e-05,  1.33370666e-04,
         2.74326721e-04,  4.75065320e-04,  6.56995892e-04,
         7.34959286e-04,  6.54435917e-04,  3.87936488e-04,
        -1.33793466e-05, -3.94908150e-04, -6.13330614e-04,
        -6.38703677e-04, -5.13236402e-04, -3.06749841e-04,
        -1.12156606e-04,  5.23564484e-06,  4.88749689e-05,
         5.65259448e-05,  5.46313742e-05,  5.18368726e-05,
         4.94707007e-05,  4.74399722e-05,  4.55192151e-05,
         4.36460799e-05,  4.18099591e-05,  3.99808010e-05,
         3.81418833e-05,  3.62168972e-05,  3.41941594e-0